# 2. Model ladder and promotion

Every rung scored by one harness on the same gold-test reviews. Rebuild with `make models`.

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # notebook-only: inline display works without it too
import matplotlib.pyplot as plt
import pandas as pd

from hotel_curator.config import Settings

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
settings = Settings(home=ROOT)

In [ ]:
import json

promotion = json.loads(
    (settings.artifacts_dir / "evals" / "promotion.json").read_text()
)
rows = pd.DataFrame(
    [
        {
            "rung": r["name"],
            "family": r["family"],
            "test macro-F1": r["summary"]["test_review_macro_f1"],
            "low": r["summary"]["test_review_macro_f1_low"],
            "high": r["summary"]["test_review_macro_f1_high"],
        }
        for r in promotion["rows"]
    ]
).sort_values("test macro-F1")
print("promoted:", promotion["winner"], "| v1 configuration:",
      promotion["v1_config"])
rows.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.45 * len(rows) + 1))
ax.errorbar(
    rows["test macro-F1"],
    rows["rung"],
    xerr=[
        rows["test macro-F1"] - rows["low"],
        rows["high"] - rows["test macro-F1"],
    ],
    fmt="o",
    color="#2a78d6",
    capsize=3,
)
ax.set_xlabel("Review-level macro-F1 on gold-test (95% CI)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

In [ ]:
winner = promotion["winner"]
reports = {
    name: json.loads(
        (settings.artifacts_dir / "evals" / name / "report.json").read_text()
    )["test"]
    for name in (winner, promotion["v1_config"])
}
pd.DataFrame(
    {name: {r["label"]: r["f1"] for r in rep["per_label"]}
     for name, rep in reports.items()}
).round(3)